# Model Extraction & Inversion: A Toy Demonstration

This notebook uses only a fictional two-feature score service. It shows how repeated queries can reveal a simple model's behavior and how aggregate outputs can disclose sensitive traits. This is a local educational simulation; do not query systems without authorization.

**Learning goals**
- Infer a toy model from input-output queries.
- See how outputs can reveal information about training data.
- Identify mitigations such as query controls, output minimization, and privacy review.

In [ ]:
# Fictional black-box model exposed as a score query.
def private_model(x):
    return round(2 * x + 1, 3)

queries = [(0.0, private_model(0.0)), (1.0, private_model(1.0))]
queries

## 1. Model extraction by querying

For this deliberately simple linear function, two exact queries determine the slope and intercept. Real models can be much more complex, but excessive access to detailed outputs may still enable approximation.

In [ ]:
(x1, y1), (x2, y2) = queries
estimated_slope = (y2 - y1) / (x2 - x1)
estimated_intercept = y1 - estimated_slope * x1

def extracted_model(x):
    return estimated_slope * x + estimated_intercept

print("Estimated parameters:", estimated_slope, estimated_intercept)
for x in (0.25, 2.5, 7.0):
    print(x, "service:", private_model(x), "extracted:", extracted_model(x))

## 2. Inference about a synthetic training group

Suppose a service returns the exact count of a sensitive attribute in a group. Subtracting overlapping group counts can reveal an individual's status. The records below are fictional; the example shows why precise aggregate outputs can leak information.

In [ ]:
# Synthetic cohort: the last record is the only member of the selected subgroup.
cohort = [
    {"person": "Demo A", "group": "general", "sensitive_trait": 0},
    {"person": "Demo B", "group": "general", "sensitive_trait": 1},
    {"person": "Demo C", "group": "general", "sensitive_trait": 0},
    {"person": "Demo D", "group": "general", "sensitive_trait": 1},
    {"person": "Demo E", "group": "selected", "sensitive_trait": 1},
]

def exact_count(rows):
    return sum(row["sensitive_trait"] for row in rows)

all_count = exact_count(cohort)
without_selected = exact_count([row for row in cohort if row["group"] != "selected"])
inferred_trait = all_count - without_selected
print("Difference of exact counts reveals selected record's trait:", inferred_trait)

## 3. Risk reduction

Apply least privilege and limit unnecessary detail. Depending on the use case, controls may include authentication, rate limits, monitoring, coarser outputs, minimum cohort sizes, and formal privacy methods. Each has utility tradeoffs and needs careful design.

In [ ]:
def coarsened_count(rows, bucket_size=3):
    count = exact_count(rows)
    lower = (count // bucket_size) * bucket_size
    return f"{lower}-{lower + bucket_size - 1}"

print("Exact count:", all_count)
print("Coarsened count:", coarsened_count(cohort))
print("Coarsening reduces precision; it is not a formal privacy guarantee.")

## Takeaways

- Model extraction approximates behavior through repeated observations; model complexity and output access affect risk.
- Inversion and membership-style inference can expose properties of training data.
- Limit access and output precision based on a documented threat model.
- Formal privacy guarantees require carefully selected methods and parameters; simple rounding alone is not differential privacy.